# Analysing Human Language using R — *talk* + *text* + *topics* workshop

✋ **NOTE** - You need to create a copy of this notebook before you work through it. This can be done by clicking on "Save a copy in Drive" option in the File menu.

<img src="https://r-talk.org/logo.png" alt="talk logo" width="200"> <img src="https://r-text.org/logo.png" alt="text logo" width="200"> <img src="https://r-topics.org/logo.png" alt="topics logo" width="200">

This notebook sets up **one shared Python environment** used by both the [talk](https://r-talk.org) package (audio: transcription, diarisation, speech embeddings) and the [text](https://r-text.org) package (text: language embeddings and analyses), and installs the [topics](https://r-topics.org) package (topic modeling of language) — so you can go from a voice recording all the way to text-based analyses and topic models in a single session.


# 0. Setup — one click

Run the single cell below (click the ▶ play button on its left) and let it finish before continuing. It downloads a pre-built environment and takes roughly **10 minutes** — a good moment to follow along with the presentation. ☕

When it is done you will see **SETUP COMPLETE** at the bottom of the output, together with a test transcription and a test text embedding.

*Optional:* to use a GPU for faster models, first select **Runtime → Change runtime type → T4 GPU** (do this before running the setup cell, because changing runtime later erases the installation).


In [ ]:
## ════════════════════════════════════════════════════════════════
##  ONE-CLICK SETUP — click the play button on this cell, then wait.
##  Takes roughly ~10 minutes. Progress messages appear below.
## ════════════════════════════════════════════════════════════════
t0 <- Sys.time()

## ── Step 1/8: System tools ───────────────────────────────────────
# ffmpeg is required for transcription (whisper loads audio through the
# ffmpeg binary). It must come from apt, NOT conda.
cat("\n=== Step 1/8: Installing system tools (ffmpeg, Java) ===\n")
system("apt-get update -qq && apt-get install -y -qq ffmpeg pigz openjdk-11-jdk-headless")

## ── Step 2/8: Miniconda via condacolab ───────────────────────────
cat("\n=== Step 2/8: Installing Miniconda (condacolab) ===\n")
system("pip install -q condacolab gdown")
system("python - <<'PY'\nimport condacolab; condacolab.install()\nPY")

## ── Step 3/8: Download the pre-built talk environment ────────────
# One archive with: the talkrpp_condaenv conda environment (torch,
# WhisNemo, WhiSPA), the talk R package + dependencies, and the
# pre-downloaded whisper/embedding models.
cat("\n=== Step 3/8: Downloading pre-built talk environment (the long step) ===\n")
file_id <- "1vU9s1vYHV96PhWWS14HQB7aJQZ59ggzE"   # talk_juli_2026.tar.gz
system(paste("gdown", file_id, "-O talk_juli_2026.tar.gz"))
stopifnot(file.exists("talk_juli_2026.tar.gz"))
cat("Unpacking…\n")
system("tar -I pigz -xf talk_juli_2026.tar.gz -C /")
unlink("talk_juli_2026.tar.gz")   # free disk space

## ── Step 4/8: Download the pre-built text R package library ──────
cat("\n=== Step 4/8: Downloading pre-built text R package library ===\n")
file_id <- "17BhUW3enbzUmKoN5nh-TASh2iP_5sS4i"   # text_juli_2026.tar.gz (built on Colab R, July 2026)
system(paste("gdown", file_id, "-O text_juli_2026.tar.gz"))
stopifnot(file.exists("text_juli_2026.tar.gz"))
untar("text_juli_2026.tar.gz", exdir = "./")
unlink("text_juli_2026.tar.gz")
.libPaths(c("library", .libPaths()))

## ── Step 5/8: Update talk (GitHub) and install topics ────────────
# The archive contains an older talk; this fetches the current release
# (pure-R install, the heavy Python stack is already in place).
cat("\n=== Step 5/8: Updating the talk R package; installing topics ===\n")
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
remotes::install_github("theharmonylab/talk", upgrade = "never", quiet = TRUE)
if (!requireNamespace("topics", quietly = TRUE)) {
  install.packages("topics", repos = "https://cloud.r-project.org")
}

## ── Step 6/8: Add the text-package Python packages ───────────────
# The same list talkrpp_install(include_text = TRUE) uses, installed
# under talk's pins so the shared environment stays consistent.
cat("\n=== Step 6/8: Installing text Python packages into talkrpp_condaenv ===\n")
conda_base <- system2("conda", c("info", "--base"), stdout = TRUE)
env_pip <- file.path(conda_base, "envs", "talkrpp_condaenv", "bin", "pip")
writeLines(c("torch==2.11.0", "torchaudio==2.11.0", "numpy==1.23.5",
             "transformers==4.57.6", "nemo-toolkit==2.7.2"),
           "talk_pins.txt")
system(paste(env_pip, "install -q -c talk_pins.txt",
             "sentence-transformers flair bertopic umap-learn hdbscan evaluate jsonschema"))

## ── Step 7/8: Wire Java into R (rJava, needed by text) ───────────
cat("\n=== Step 7/8: Configuring Java for R ===\n")
java_home <- dirname(dirname(system2("readlink", c("-f", Sys.which("javac")), stdout = TRUE)))
Sys.setenv(
  JAVA_HOME       = java_home,
  LD_LIBRARY_PATH = paste(file.path(java_home, "lib/server"),
                          Sys.getenv("LD_LIBRARY_PATH"), sep = ":")
)
system("R CMD javareconf")
dyn.load(file.path(java_home, "lib/server/libjvm.so"))
rjava_ok <- tryCatch({ library(rJava); TRUE }, error = function(e) {
  message("Pre-built rJava incompatible with this R — recompiling from source…")
  FALSE
})
if (!rjava_ok) {
  install.packages("rJava", repos = "https://cloud.r-project.org")
  library(rJava)
}
.jinit()   # 0 means the JVM loaded without errors

## ── Step 8/8: Initialize BOTH packages to the shared environment ─
cat("\n=== Step 8/8: Initializing talk and text (shared environment) ===\n")
library(reticulate)
Sys.setenv(RETICULATE_MINICONDA_PATH = system2("conda", c("info", "--base"), stdout = TRUE))
talk::talkrpp_initialize()
library(talk)
text::textrpp_initialize(condaenv = "talkrpp_condaenv", save_profile = FALSE)
library(text)
library(topics)

# Quick proof that BOTH work in the same session (models are already
# downloaded, so this should only take seconds)
wav <- system.file("extdata/test_short.wav", package = "talk")
transcription <- talkText(wav)
print(transcription)
emb <- textEmbed(transcription$transcription, model = "prajjwal1/bert-tiny")
cat("textEmbed dimensions:", dim(emb$texts[[1]]), "\n")

## ── Done ─────────────────────────────────────────────────────────
cat("\n============================================================\n")
cat("  SETUP COMPLETE in",
    round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1),
    "minutes — talk, text and topics are loaded and ready!\n")
cat("============================================================\n")


# 1. From voice to text

The `talk` package ships with a short example recording, so you can try it without uploading anything.


In [ ]:
# A short example recording bundled with the package
wav_path <- system.file("extdata/test_short.wav", package = "talk")
wav_path

# Speech-to-text transcription: a tibble with one row per file
transcription <- talkText(wav_path)
transcription


# 2. Speech embeddings (talk)

Embeddings represent the *sound* of the recording (voice acoustics, prosody) as numbers, ready for downstream analyses.


In [ ]:
embeddings_audio <- talkEmbed(wav_path)
dim(embeddings_audio)
embeddings_audio[, 1:5]


# 3. From voice to *language* analysis — talk + text together

This is where the shared environment pays off: transcribe speech with **talk**, then analyse the words with **text** — in the same R session.

(The example uses a small, fast model; drop the `model` argument to use text's default.)


In [ ]:
# Language embeddings of the transcribed words (text package)
embeddings_text <- textEmbed(
  transcription$transcription,
  model = "prajjwal1/bert-tiny"
)
embeddings_text$texts[[1]]


# 4. Who said what? (speaker diarisation)

For conversations, `talkTranscribeDiarise()` additionally separates the speakers. The example below uses a bundled two-speaker recording — this runs the full diarisation pipeline, so expect a few minutes on CPU (faster on GPU).


In [ ]:
wav_two_speakers <- system.file("extdata/test_diarise.wav", package = "talk")
conversation <- talkTranscribeDiarise(wav_two_speakers, num_speakers = 2)
conversation


# 5. Try your own recording

Upload a `.wav` file using the folder icon 📁 in the left sidebar (drag and drop), then point the functions at it, e.g.:

```r
my_transcript <- talkText("my_recording.wav")
my_embeddings <- textEmbed(my_transcript$transcription)
```

The `topics` package is also installed and loaded — see the tutorials on [r-topics.org](https://r-topics.org) for topic modeling of your transcribed language.
